# Total Checkup EDA

공복혈당 기준 당뇨 상태 전이 예측을 위한 탐색적 데이터 분석(EDA)입니다.

| 사용 데이터 | 설명 |
|------------|------|
| `adoc_v1.total_checkups.json` | 건강검진 기록 (공복혈당 포함) |
| `adoc_v1.medical_histories.json` | 처방 의약품 이력 |

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("..").resolve()))

from core.loader import DataLoader
from core.eda import CheckupDateAnalyzer

OUTPUT_DIR = Path("../outputs/260522_EDA")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 1. 검진일 분포 (Checkup Date Distribution)

전체 검진 건수·인원·기간 개요 및 연도/월별 분포를 확인합니다.

In [2]:
EXCLUDE_USER_KEYS = [
    "INVALID_RESULT",  # user_key가 유효하지 않은 이상치 (32회 검진)
]

df = DataLoader("adoc_v1.total_checkups.json", exclude_user_keys=EXCLUDE_USER_KEYS).load()
analyzer = CheckupDateAnalyzer(df, date_col="checkup_date")

In [3]:
national_df = DataLoader("adoc_v1.national_checkups.json").load_streaming(
    columns=["user_key", "checkup_date"]
)
national_analyzer = CheckupDateAnalyzer(national_df, date_col="checkup_date")

def _print_summary(label: str, s: dict) -> None:
    print(f"[{label}]")
    print(f"  검진 건수 : {s['count']:,}건")
    print(f"  검진 인원 : {s['user_count']:,}명")
    print(f"  최초 검진일: {s['min'].strftime('%Y-%m-%d')}")
    print(f"  최근 검진일: {s['max'].strftime('%Y-%m-%d')}")
    print(f"  포함 연도  : {s['unique_years']}")
    print()

_print_summary("종합검진", analyzer.summary())
_print_summary("국가검진", national_analyzer.summary())

/data2/mason/prediabetes_diabetes/core/loader.py:53: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  converted = pd.to_datetime(df[col], utc=True, errors="coerce")


[종합검진]
  검진 건수 : 35,833건
  검진 인원 : 29,597명
  최초 검진일: 2019-09-10
  최근 검진일: 2026-02-28
  포함 연도  : [2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026]

[국가검진]
  검진 건수 : 202,881건
  검진 인원 : 51,864명
  최초 검진일: 2013-01-01
  최근 검진일: 2026-03-19
  포함 연도  : [2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026]



In [4]:
analyzer.plot_distribution(output_dir=OUTPUT_DIR, show=False)
national_analyzer.plot_distribution(
    output_dir=OUTPUT_DIR,
    show=False,
    filename="national_checkup_date_distribution.png",
)

/data2/mason/prediabetes_diabetes/core/eda.py:84: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  month_counts = dates.dt.to_period("M").value_counts().sort_index()


Saved: ../outputs/260522_EDA/total_checkup_date_distribution.png


/data2/mason/prediabetes_diabetes/core/eda.py:84: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  month_counts = dates.dt.to_period("M").value_counts().sort_index()


Saved: ../outputs/260522_EDA/national_checkup_date_distribution.png


## 2. 반복 검진 분석 (Repeated Checkup Analysis)

두 번 이상 검진을 받은 환자 현황과 재방문 주기 분포를 분석합니다.

In [5]:
from core.interval import CheckupIntervalAnalyzer

interval_analyzer = CheckupIntervalAnalyzer(df, date_col="checkup_date", user_col="user_key")

In [6]:
s = interval_analyzer.summary()
print(f"전체 환자 수       : {s['total_patients']:,}명")
print(f"1회만 검진         : {s['single_visit']:,}명")
print(f"2회 이상 검진      : {s['repeat_visit']:,}명  ({s['repeat_ratio']}%)")
print(f"최다 검진 횟수     : {s['max_visits']}회")
print()
print(f"재방문 주기 중앙값 : {s['median_interval_days']}일 ({s['median_interval_days']/30.44:.1f}개월)")
print(f"재방문 주기 평균   : {s['mean_interval_days']}일 ({s['mean_interval_days']/30.44:.1f}개월)")
print(f"IQR               : {s['interval_q1']}일 ~ {s['interval_q3']}일")

전체 환자 수       : 29,597명
1회만 검진         : 24,811명
2회 이상 검진      : 4,786명  (16.2%)
최다 검진 횟수     : 5회

재방문 주기 중앙값 : 371.0일 (12.2개월)
재방문 주기 평균   : 439.2일 (14.4개월)
IQR               : 336.0일 ~ 448.0일


In [7]:
interval_analyzer.plot_analysis(output_dir=OUTPUT_DIR, show=False)

Saved: ../outputs/260522_EDA/repeated_checkup_analysis.png


## 3. 결측치 분석 (Missing Value Analysis)

2회 이상 검진을 받은 수검자를 대상으로 공복혈당·당화혈색소 결측 현황을 확인합니다.

In [8]:
from core.missing import MissingValueAnalyzer

missing_analyzer = MissingValueAnalyzer(df, min_visits=2)

national_missing_df = DataLoader("adoc_v1.national_checkups.json").load_streaming(
    columns=["user_key", "checkup_date", "detail_infos"]
)
national_missing_analyzer = MissingValueAnalyzer(
    national_missing_df,
    fields={"CH164": "공복혈당"},
    min_visits=2,
)

/data2/mason/prediabetes_diabetes/core/loader.py:53: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  converted = pd.to_datetime(df[col], utc=True, errors="coerce")


In [9]:
# print("[종합검진]")
# display(missing_analyzer.summary())
# print("[국가검진]")
# display(national_missing_analyzer.summary())

In [10]:
# MissingValueAnalyzer.plot_missing_comparison(
#     sources=[
#         ("종합검진", missing_analyzer),
#         ("국가검진", national_missing_analyzer),
#     ],
#     output_dir=OUTPUT_DIR,
#     show=False,
# )

## 4. 당뇨 상태 전이 분석 (Glucose State Transition)

공복혈당 기준으로 상태를 분류하고, 종합·국가 검진을 결합한 수검자의 검진 간 전이 흐름을 분석합니다.

| 상태 | 공복혈당 기준 |
|------|--------------|
| 정상 | 100 미만 |
| 당뇨병전단계 | 100 이상 125 이하 |
| 당뇨병 | 125 초과 |

In [11]:
from core.transition import CombinedGlucoseTransitionAnalyzer

# national_missing_df: 섹션 3에서 로드한 국가검진 DataFrame (user_key, checkup_date, detail_infos)
transition_analyzer = CombinedGlucoseTransitionAnalyzer(df, national_missing_df)

In [12]:
import pandas as pd
pd.set_option("display.max_rows", None)

print("=== 전체 전이 요약 ===")
display(transition_analyzer.summary())

seq_dict = transition_analyzer.sequence_summary(n_visits=[2, 3, 4, 5])
for n, df in seq_dict.items():
    sep = "=" * 30
    print(sep)
    print(f"{n}회 수검자 시퀀스")
    print(sep)
    display(df)


=== 전체 전이 요약 ===


,from_state,to_state,count,label,pct
0,정상,정상,94450,정상 → 정상,56.6
1,당뇨병전단계,당뇨병전단계,22812,당뇨병전단계 → 당뇨병전단계,13.7
2,정상,당뇨병전단계,21345,정상 → 당뇨병전단계,12.8
3,당뇨병전단계,정상,18202,당뇨병전단계 → 정상,10.9
4,당뇨병,당뇨병,4186,당뇨병 → 당뇨병,2.5
5,당뇨병전단계,당뇨병,2566,당뇨병전단계 → 당뇨병,1.5
6,당뇨병,당뇨병전단계,1978,당뇨병 → 당뇨병전단계,1.2
7,정상,당뇨병,640,정상 → 당뇨병,0.4
8,당뇨병,정상,556,당뇨병 → 정상,0.3


2회 수검자 시퀀스


,sequence,count,pct
0,정상 → 정상,6186,61.5
1,정상 → 당뇨병전단계,1259,12.5
2,당뇨병전단계 → 당뇨병전단계,1089,10.8
3,당뇨병전단계 → 정상,1017,10.1
4,당뇨병 → 당뇨병,204,2.0
5,당뇨병전단계 → 당뇨병,134,1.3
6,당뇨병 → 당뇨병전단계,81,0.8
7,정상 → 당뇨병,49,0.5
8,당뇨병 → 정상,38,0.4


3회 수검자 시퀀스


,sequence,count,pct
0,정상 → 정상 → 정상,4614,54.6
1,정상 → 정상 → 당뇨병전단계,741,8.8
2,정상 → 당뇨병전단계 → 정상,558,6.6
3,당뇨병전단계 → 정상 → 정상,544,6.4
4,정상 → 당뇨병전단계 → 당뇨병전단계,453,5.4
5,당뇨병전단계 → 당뇨병전단계 → 당뇨병전단계,446,5.3
6,당뇨병전단계 → 정상 → 당뇨병전단계,306,3.6
7,당뇨병전단계 → 당뇨병전단계 → 정상,287,3.4
8,당뇨병 → 당뇨병 → 당뇨병,111,1.3
9,당뇨병전단계 → 당뇨병전단계 → 당뇨병,60,0.7


4회 수검자 시퀀스


,sequence,count,pct
0,정상 → 정상 → 정상 → 정상,3651,45.9
1,정상 → 정상 → 정상 → 당뇨병전단계,492,6.2
2,정상 → 정상 → 당뇨병전단계 → 정상,417,5.2
3,정상 → 당뇨병전단계 → 정상 → 정상,399,5.0
4,당뇨병전단계 → 정상 → 정상 → 정상,373,4.7
5,당뇨병전단계 → 당뇨병전단계 → 당뇨병전단계 → 당뇨병전단계,337,4.2
6,정상 → 정상 → 당뇨병전단계 → 당뇨병전단계,310,3.9
7,정상 → 당뇨병전단계 → 당뇨병전단계 → 당뇨병전단계,229,2.9
8,정상 → 당뇨병전단계 → 정상 → 당뇨병전단계,191,2.4
9,정상 → 당뇨병전단계 → 당뇨병전단계 → 정상,182,2.3


5회 수검자 시퀀스


,sequence,count,pct
0,정상 → 정상 → 정상 → 정상 → 정상,2629,39.9
1,정상 → 정상 → 정상 → 정상 → 당뇨병전단계,310,4.7
2,정상 → 당뇨병전단계 → 정상 → 정상 → 정상,274,4.2
3,정상 → 정상 → 정상 → 당뇨병전단계 → 정상,254,3.9
4,정상 → 정상 → 당뇨병전단계 → 정상 → 정상,253,3.8
5,당뇨병전단계 → 정상 → 정상 → 정상 → 정상,227,3.4
6,당뇨병전단계 → 당뇨병전단계 → 당뇨병전단계 → 당뇨병전단계 → 당뇨병전단계,202,3.1
7,정상 → 정상 → 정상 → 당뇨병전단계 → 당뇨병전단계,172,2.6
8,정상 → 당뇨병전단계 → 당뇨병전단계 → 당뇨병전단계 → 당뇨병전단계,137,2.1
9,정상 → 정상 → 당뇨병전단계 → 당뇨병전단계 → 당뇨병전단계,127,1.9


In [13]:
transition_analyzer.plot_sequences(n_visits=[2, 3, 4, 5], top_n=10, output_dir=OUTPUT_DIR, show=False)

Saved: ../outputs/260522_EDA/glucose_sequences.png


## 5. Json 생성

공복혈당 기준 상태 전이, 공복혈당 변화량, 당뇨병용제 처방 이력을 포함한 모델 학습용 JSON 파일로 저장합니다.

| 포함 필드 | 설명 |
|-----------|------|
| `current_glucose` / `future_glucose` | 현재·다음 검진 시점 공복혈당 (mg/dL) |
| `glucose_change` | 공복혈당 변화량 (`future − current`) |
| `selected_transition` / `full_transition` | 이력 출처를 괄호로 표기 (예: `정상(종합) → 정상(국가)`) |
| `diabetes_drugs` | 처방 약물 목록 (`drug_name`, `start_date`) |

### 5-1. 약물 이력 필터 설정

`adoc_v1.medical_histories.json`에서 종합·국가 검진 결합 대상 수검자의 처방 이력을 조회합니다.  
`DRUG_EFFECTS` 리스트를 수정하여 포함할 약효 분류를 변경할 수 있습니다.

In [14]:
from core.loader import MedicalHistoryLoader

# ── 약물 필터 설정 ──────────────────────────────────────────
# JSON에 포함할 drug_effect 목록 (여러 개 지정 가능)
DRUG_EFFECTS = ["당뇨병용제"]
# ────────────────────────────────────────────────────────────

# 2회 이상 수검자에 대해서만 의료 이력 조회
drug_lookup = MedicalHistoryLoader(drug_effects=DRUG_EFFECTS).load_drug_lookup(
    user_keys=transition_analyzer.user_keys
)
print(f"당뇨병용제 처방 이력 보유 사용자: {len(drug_lookup):,}명 / 대상 수검자: {len(transition_analyzer.user_keys):,}명")

당뇨병용제 처방 이력 보유 사용자: 2,517명 / 대상 수검자: 71,250명


### 5-2. 2회 수검자 데이터셋

종합·국가 검진 결합 기준으로 정확히 **2회** 수검한 수검자 중, **첫 검진이 정상** 또는 **당뇨병전단계**인 수검자를 아래 기준으로 분류하여 JSON으로 저장합니다.  
`selected_transition` / `full_transition` 에는 이력 출처(종합/국가)가 괄호로 표기됩니다.

| dataset | label=0 (음성) | label=1 (양성) |
|---------|--------------|---------------|
| `pre-diabetes` | 정상 → 정상 | 정상 → 당뇨병전단계 / 정상 → 당뇨병 |
| `diabetes` | 당뇨병전단계 → 당뇨병전단계 (또는 정상) | 당뇨병전단계 → 당뇨병 |

In [15]:
dataset_df = transition_analyzer.export_dataset(
    output_path=OUTPUT_DIR / "two_visit_dataset.json",
    drug_lookup=drug_lookup,
)

for ds in ["pre-diabetes", "diabetes"]:
    sub = dataset_df[dataset_df["dataset"] == ds]
    print(f"[{ds}] 총 {len(sub):,}명")
    print(sub["label"].value_counts().rename({0: "label=0", 1: "label=1"}).to_string())
    print()

Saved: ../outputs/260522_EDA/two_visit_dataset.json  (9,727명)
[pre-diabetes] 총 7,487명
label
label=0    6180
label=1    1307

[diabetes] 총 2,240명
label
label=0    2106
label=1     134



### 5-3. 3회 이상 수검자 데이터셋

종합·국가 검진 결합 기준으로 **3회 이상** 수검한 수검자를 아래 기준으로 분류합니다.  
`selected_transition` / `full_transition` 에는 이력 출처(종합/국가)가 괄호로 표기됩니다.

인접한 (i, i+1) 쌍 외에도, 중간 스텝을 포함하는 **(i, j) 다중 스텝 쌍**을 label=0으로 추가합니다.  
예: 방문 1→2→3→4가 있으면 `selected_transition`에 `2→3→4` 케이스도 포함됩니다.

**분류 기준**

| dataset | label | 단일 스텝 조건 | 다중 스텝 추가 조건 |
|---------|-------|--------------|------------------|
| pre-diabetes | 0 | 정상→정상 (**과거 전체**에 전단계/당뇨 이력 없는 경우) | 중간 상태 전부 정상 |
| pre-diabetes | 1 | 정상→전단계/당뇨 (**과거 전체**에 전단계/당뇨 이력 없는 경우) | *(다중 스텝 미적용)* |
| diabetes | 0 | 전단계→전단계/정상 (**과거 전체**에 당뇨 이력 없는 경우) | 중간 상태에 당뇨 없음 |
| diabetes | 1 | 전단계→당뇨 (**과거 전체**에 당뇨 이력 없는 경우) | *(다중 스텝 미적용)* |

동일 수검자에게 유효한 전이 쌍이 여러 개 존재하는 경우 **모두 포함**합니다.  
JSON 키는 `{dataset}::{user_key}::{current_checkup_date}::{future_checkup_date}` 형태로 고유성을 보장합니다.

In [16]:
multi_result = transition_analyzer.export_multi_visit_dataset(
    min_visits=3,
    output_path=OUTPUT_DIR / "multi_visit_dataset.json",
    drug_lookup=drug_lookup,
)

[pre-diabetes] 총 205,185건  (label=0: 192,621, label=1: 12,564)
[diabetes] 총 113,824건  (label=0: 112,205, label=1: 1,619)
Saved: ../outputs/260522_EDA/multi_visit_dataset.json
